In [ ]:
import pandas as pd
import requests
import time
import os
import re
import threading
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Config ---
OUTPUT_FILE = 'movies_wiki_enriched.csv'
WORKERS = 3               # stay well under rate limit
DELAY = 0.3               # per-thread base delay
CHECKPOINT_EVERY = 200
MAX_RETRIES = 4
MIN_INTERVAL = 0.4        # global min seconds between requests (~150 req/min ceiling)

lock = threading.Lock()

# --- Global rate limiter ---
_last_request_time = 0.0
_rate_lock = threading.Lock()

def rate_limited_sleep():
    """Ensure at least MIN_INTERVAL seconds between any two requests, globally."""
    global _last_request_time
    with _rate_lock:
        now = time.monotonic()
        wait = MIN_INTERVAL - (now - _last_request_time)
        if wait > 0:
            time.sleep(wait)
        _last_request_time = time.monotonic()

# --- Load dataset ---
df = pd.read_csv('top_10k_movies_2000_min_1m_filtered.csv')

# --- Checkpoint ---
if os.path.exists(OUTPUT_FILE):
    done = pd.read_csv(OUTPUT_FILE)
    done_ids = set(done['tmdb_id'].tolist())
    print(f"Resuming — {len(done_ids)} already done")
else:
    done = pd.DataFrame()
    done_ids = set()

# --- Helpers ---
def get_year(release_date):
    try:
        return str(pd.to_datetime(release_date).year)
    except:
        return ""

HEADERS = {'User-Agent': 'MovieResearch/1.0 (your@email.com)'}

def safe_get(params, retries=MAX_RETRIES):
    """Rate-limited GET with exponential backoff."""
    for attempt in range(retries):
        rate_limited_sleep()
        try:
            resp = requests.get(
                "https://en.wikipedia.org/w/api.php",
                params=params,
                headers=HEADERS,
                timeout=15
            )
            if resp.status_code == 429 or not resp.text.strip():
                wait = 2 ** attempt + random.uniform(0, 1)
                print(f"  ⚠ Rate limited, waiting {wait:.1f}s...")
                time.sleep(wait)
                continue
            return resp.json()
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            wait = 2 ** attempt + random.uniform(0, 1)
            print(f"  ⚠ Connection error ({e}), waiting {wait:.1f}s...")
            time.sleep(wait)
        except Exception:
            time.sleep(2 ** attempt)
    return None

def wiki_search(query):
    data = safe_get({
        "action": "query", "list": "search",
        "srsearch": query, "srlimit": 1, "format": "json"
    })
    if not data or "error" in data:
        return None
    results = data.get("query", {}).get("search", [])
    return results[0]["title"] if results else None

def wiki_get_sections(page_title):
    data = safe_get({
        "action": "parse", "page": page_title,
        "prop": "sections|wikitext", "format": "json"
    })
    if not data or "error" in data or "parse" not in data:
        return None, None, None
    sections = data["parse"]["sections"]
    wikitext = data["parse"]["wikitext"]["*"]
    fullurl = f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"
    return sections, wikitext, fullurl

def extract_section_text(wikitext, section_title):
    pattern = rf'==+\s*{re.escape(section_title)}\s*==+'
    parts = re.split(r'(==+[^=]+==+)', wikitext)
    found = False
    texts = []
    current_level = None
    for part in parts:
        if re.match(r'==+[^=]+==+', part):
            level = len(re.match(r'(=+)', part).group(1))
            if re.match(pattern, part, re.IGNORECASE):
                found = True
                current_level = level
                continue
            elif found and level <= current_level:
                break
        elif found:
            clean = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]]*)\]\]', r'\1', part)
            clean = re.sub(r'\{\{[^}]*\}\}', '', clean)
            clean = re.sub(r'<[^>]+>', '', clean)
            clean = re.sub(r'\[https?://[^\]]*\]', '', clean)
            clean = re.sub(r"'{2,3}", '', clean)
            clean = re.sub(r'\n{3,}', '\n\n', clean).strip()
            if clean:
                texts.append(clean)
    return '\n\n'.join(texts) if texts else None

def wiki_batch_lookup(candidates):
    """
    Look up multiple titles in ONE API call using pipe-separated titles.
    Returns the first non-missing page title found, or None.
    """
    titles_param = "|".join(candidates)
    data = safe_get({
        "action": "query", "titles": titles_param,
        "redirects": 1, "format": "json"
    })
    if not data or "error" in data:
        return None
    pages = data.get("query", {}).get("pages", {})
    # Preserve candidate priority order
    title_to_page = {}
    for page in pages.values():
        if "missing" not in page:
            title_to_page[page.get("title", "").lower()] = page["title"]
    # Return first candidate that matched
    for c in candidates:
        key = c.lower()
        if key in title_to_page:
            return title_to_page[key]
        # also check redirect targets
        for resolved_title in title_to_page.values():
            return resolved_title  # return first non-missing result
    return None

def get_wiki_data(title, release_date, tmdb_id):
    # Random jitter to desync parallel threads at start
    time.sleep(random.uniform(0, 0.3))

    year = get_year(release_date)

    # Step 1: Batch lookup — all 3 candidates in ONE request
    candidates = [f"{title} ({year} film)", f"{title} film", title]
    page_title = wiki_batch_lookup(candidates)

    # Step 2: Search fallback (only if batch found nothing)
    if page_title is None:
        page_title = wiki_search(f"{title} {year} film")

    if page_title is None:
        return {
            "tmdb_id": tmdb_id, "title": title, "wiki_found": False,
            "wiki_title": None, "wiki_url": None,
            "wiki_plot": None, "wiki_critical_response": None,
        }

    # Step 3: Fetch sections
    sections, wikitext, fullurl = wiki_get_sections(page_title)
    if sections is None:
        return {
            "tmdb_id": tmdb_id, "title": title, "wiki_found": False,
            "wiki_title": page_title, "wiki_url": None,
            "wiki_plot": None, "wiki_critical_response": None,
        }

    section_titles = [s["line"] for s in sections]

    # Step 4: Plot
    plot_text = None
    for candidate in ["Plot", "Synopsis", "Storyline", "Story"]:
        match = next((s for s in section_titles if s.lower() == candidate.lower()), None)
        if match:
            plot_text = extract_section_text(wikitext, match)
            if plot_text:
                break

    # Step 5: Critical response
    critical_text = None
    for candidate in ["Critical response", "Critical reception", "Reviews",
                      "Critical and public response", "Response", "Reception"]:
        match = next((s for s in section_titles if s.lower() == candidate.lower()), None)
        if match:
            critical_text = extract_section_text(wikitext, match)
            if critical_text:
                break

    return {
        "tmdb_id": tmdb_id, "title": title, "wiki_found": True,
        "wiki_title": page_title, "wiki_url": fullurl,
        "wiki_plot": plot_text, "wiki_critical_response": critical_text,
    }

# --- Parallel execution ---
to_process = df[~df['tmdb_id'].isin(done_ids)].to_dict('records')
print(f"Processing {len(to_process)} movies with {WORKERS} workers...")

results = []
completed = 0

def process_row(row):
    return get_wiki_data(row['title'], row['release_date'], row['tmdb_id'])

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_row, row): row for row in to_process}
    for future in as_completed(futures):
        try:
            result = future.result()
            with lock:
                results.append(result)
                completed += 1
                status = "✓" if result['wiki_found'] else "✗"
                print(f"[{completed}/{len(to_process)}] {status} {result['title']}")
                if completed % CHECKPOINT_EVERY == 0:
                    chunk = pd.DataFrame(results)
                    combined = pd.concat([done, chunk], ignore_index=True)
                    combined.to_csv(OUTPUT_FILE, index=False)
                    print(f" Checkpoint saved at {completed} rows")
        except Exception as e:
            print(f"  Error: {e}")

# --- Final save ---
final = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)

merged = df.merge(
    final[['tmdb_id', 'wiki_found', 'wiki_title', 'wiki_url', 'wiki_plot', 'wiki_critical_response']],
    on='tmdb_id', how='left'
)
merged.to_csv('movies_full_enriched.csv', index=False)

found = final['wiki_found'].sum()
print(f"\nDone! {found}/{len(final)} movies found ({found/len(final)*100:.1f}%)")


Processing 3781 movies with 3 workers...
[1/3781] ✓ Avatar
[2/3781] ✓ Avatar: The Way of Water
[3/3781] ✓ Avengers: Endgame
[4/3781] ✓ Star Wars: The Force Awakens
[5/3781] ✓ Avengers: Infinity War
[6/3781] ✓ Spider-Man: No Way Home
[7/3781] ✓ Zootopia 2
[8/3781] ✓ Jurassic World
[9/3781] ✓ Inside Out 2
[10/3781] ✓ The Lion King
[11/3781] ✓ Furious 7
[12/3781] ✓ Top Gun: Maverick
[13/3781] ✓ Frozen II
[14/3781] ✓ Barbie
[15/3781] ✓ The Avengers
[16/3781] ✓ The Super Mario Bros. Movie
[17/3781] ✓ Black Panther
[18/3781] ✓ Avatar: Fire and Ash
[19/3781] ✓ Avengers: Age of Ultron
[20/3781] ✓ Jurassic World: Fallen Kingdom
[21/3781] ✓ Deadpool & Wolverine
[22/3781] ✓ Frozen
[23/3781] ✓ Beauty and the Beast
[24/3781] ✓ Minions
[25/3781] ✓ Incredibles 2
[26/3781] ✓ The Lord of the Rings: The Return of the King
[27/3781] ✓ Star Wars: The Last Jedi
[28/3781] ✓ Joker
[29/3781] ✓ Aquaman
[30/3781] ✓ Iron Man 3
[31/3781] ✓ The Fate of the Furious
[32/3781] ✓ Captain Marvel
[33/3781] ✓ Spider-Man: